# SQLite Database Setup

These scripts are intended to be run initially to ensure the structure of the Database is created correctly.

This means:

- Creating the Database itself
- Creating all Tables on the Database
    - Ensuring each table has Foreign Keys created
    - Ensuring all tables have necessary indexes created

## Set Local Variables

In [1]:
import json
import pandas as pd

db_name = 'DiscordChatMusic.db'
local_table_definitions = 'sqlite_resources/DDL/TABLE_CREATION/'

# Order of table creation is manually designed. This is intentional as this is a Star Schema setup. So Dimensions must be created before Facts
local_tables = [
    'discord_logging.sql',          # Just for logging
    'discord_guilds.sql',
    'discord_channels.sql',         # References Guilds
    'discord_authors.sql',
    'discord_author_roles.sql',     # References Authors
    'discord_author_nicknames.sql', # References Authors
    'discord_messages.sql',         # References Authors / Channels / Guilds
    'discord_mentions.sql',         # References Authors / Messages
    'discord_reaction_authors.sql', # References Authors / Messages
    'discord_attachments.sql',      # References Messages
    'discord_reactions.sql',        # References Messages
    'discord_stickers.sql',         # References Messages
    'discord_urls.sql',             # References Messages
    'discord_embeds.sql'            # References Messages
]

json_file = "tunes.json"

with open(json_file, 'r') as j:
    contents = json.loads(j.read())

In [2]:
import sqlite3

# Creates a new database if doesn't exist
conn = sqlite3.connect(db_name)

for local_table in local_tables:
    file_url = local_table_definitions + local_table
    with open(file_url, 'r') as sql_file:
        try:
            conn.executescript(sql_file.read())
        except sqlite3.OperationalError:
            print(f"OperationalError: On {local_table} at location {file_url}")

#conn.close()

# Filling Tables

In [3]:
from typing import List

def comma_list_surrounded_by_character(l: List[str], c: str):

    return c + f"{c},{c}".join(l) + c

# TODO: name no longer representive of what method accomplishes
def values_in_table(table: str, filter_col: str, filter_condition: List[str], conn: sqlite3.Connection, not_flag=False) -> bool:

    cur = conn.cursor()

    if not_flag:
        not_string = 'NOT'
    else:
        not_string = ''

    select_sql = f"""
        SELECT 
            {filter_col}
        FROM {table}
        WHERE {filter_col} 
            -- Take a Python list and turn it into a list of SQL strings
            {not_string} IN ({comma_list_surrounded_by_character(l=filter_condition,c="'")})
    """

    cur.execute(select_sql)

    result = cur.fetchall()

    #print(result)

    cur.close()

    # No matches? Return -1
    if len(result) == 0:
        return -1

    # Convert tuple -> List
    return [i[0] for i in result]

In [4]:
from datetime import datetime

def todays_datetime():
    """
    Returns today's Date as a string with the format YYYY-MM-DD HH:MM:SS
    """

    return datetime.today().strftime('%Y-%m-%d %H:%M:%S')

# TODO: Make dynamic
def logging_process():

    return "sqlite_database_setup.ipynb"

def sqlite_insert_log_single_success(conn: sqlite3.Connection, table_name: str, logging_table: str, logging_table_id: str, file_name: str):

    cur = conn.cursor()

    insert_sql = f"""
    INSERT INTO {table_name}(logging_date, logging_table, logging_table_id, logging_file_name, logging_process)
    VALUES('{todays_datetime()}', '{logging_table}','{logging_table_id}','{file_name}','{logging_process()}');
    """

    cur.execute(insert_sql)

    conn.commit()
    cur.close()

    return


def sqlite_insert_single_row_df(json_blob: str, inner_key: str, mapping: dict, table_name: str, table_primary_key: str, conn: sqlite3.Connection, file_name: str) -> None:

    df = pd.DataFrame([json_blob[inner_key]])

    assert len(df) == 1, f"sqlite_insert_single_row_df is designed for single items. This instead has {len(df)} items."

    # Filter DataFrame to only be mapped columns, if the col doesn't exist in the Database we are not interested.
    df = df[mapping.keys()]

    df.rename(columns=mapping, inplace=True)

    ids = df[table_primary_key].to_list()

    table_values = values_in_table(
        table=table_name,
        filter_col=table_primary_key,
        filter_condition=ids,
        conn=conn, 
        not_flag=False
    )

    id = ids[0]

    # Single value not inserted into table on DB
    if table_values == -1:
        df.to_sql(table_name,conn,if_exists='append',index=False)
        sqlite_insert_log_single_success(
            conn=conn,
            table_name='discord_logging',
            logging_table=table_name,
            file_name=file_name,
            logging_table_id=id
        )
        print(f"{df.head(1)} inserted into {table_name} successfully.")
    else:
        print(f"{table_values} already exists within {table_name}, nothing inserted.")

    return id

## discord_guilds

In [67]:
guild_id = sqlite_insert_single_row_df(
    json_blob=contents,
    inner_key="guild",
    mapping={
        "id": "guild_id",
        "name": "guild_name",
        "iconUrl": "guild_icon_URL"
    },
    table_name='discord_guilds',
    table_primary_key='guild_id',
    conn=conn,
    file_name=json_file
)

['1185399888880619550'] already exists within discord_guilds, nothing inserted.


## discord_channels

In [68]:
channel_id = sqlite_insert_single_row_df(
    json_blob=contents,
    inner_key="channel",
    table_name='discord_channels',
    table_primary_key='channel_id',
    mapping={
        "id": "channel_id",
        "type": "channel_type",
        "categoryId": "guild_id",
        "category": "channel_category",
        "name": "channel_name"
    },
    conn=conn,
    file_name=json_file
)

['1203039998740271166'] already exists within discord_channels, nothing inserted.


## discord_messages

In [42]:
def sqlite_insert_log_multiple_success(conn: sqlite3.Connection, table_name: str, logging_table: str, logging_table_ids: List[str], file_name: str):

    num_ids = len(logging_table_ids)

    df_dict = {
        "logging_date": [todays_datetime()] * num_ids,
        "logging_table": [table_name] * num_ids,
        "logging_table_id": logging_table_ids,
        "logging_file_name": [file_name] * num_ids,
        "logging_process": [logging_process()] * num_ids
    }

    insert_df = pd.DataFrame(df_dict)

    insert_df.to_sql(logging_table,conn,if_exists='append',index=False)

    return

# TODO: If logging fails, items are still inserted, think about options
def insert_df_to_db(df: pd.DataFrame, table_name: str, logging_table: str, conn: sqlite3.Connection, file_name: str, ids: List[str]) -> None:
    df.to_sql(table_name,conn,if_exists='append',index=False)
    sqlite_insert_log_multiple_success(
        conn=conn,
        table_name=table_name,
        logging_table=logging_table,
        file_name=file_name,
        logging_table_ids=ids
    )
    return

def assign_constant_column_values(df: pd.DataFrame, constant_cols) -> pd.DataFrame:

    assert len(constant_cols.keys()) == len(constant_cols.values()), "Dictionary object assumed to have a 1-to-1 mapping of keys to values."

    for key, value in constant_cols.items():
        df[key] = value

    return df

# TODO: Remove overlap betwen this and single row and think about naming updates and readability for how many methods there are
def sqlite_insert_multiple_row_df(
        json_normalize_dict: dict
        , mapping: dict
        , table_name: str
        , table_primary_key: str
        , conn: sqlite3.Connection
        , file_name: str
        , constant_cols={}
        , drop_duplicates=False
    ) -> None:

    df = pd.json_normalize(**json_normalize_dict)

    if df.empty:
        print("No Records to Insert.")
        return -1

    df = assign_constant_column_values(df=df, constant_cols=constant_cols)

    # Filter DataFrame to only be mapped columns, if the col doesn't exist in the Database we are not interested.
    df = df[mapping.keys()]

    df.rename(columns=mapping, inplace=True)

    ids = df[table_primary_key].to_list()

    table_values = values_in_table(
        table=table_name,
        filter_col=table_primary_key,
        filter_condition=ids,
        conn=conn, 
        not_flag=False
    )

    if drop_duplicates:
        df = df.drop_duplicates()

    # ALL values are not present on DataBase, we are good to insert everything
    # Unecessary to check because we are attempting to insert as a default, exiting early only if there is nothing to insert
    #if table_values == -1:

    # At-least one record from the DataFrame exists within the DataBase
    if table_values != -1:
        # Filter to records that are not present in the DataBase
        df = df[~df[table_primary_key].isin(table_values)]
    # Nothing to insert, leave
    if df.empty:
        print("Nothing to insert.")
        return ids
    
    insert_df_to_db(
        df=df,
        conn=conn,
        table_name=table_name,
        logging_table='discord_logging',
        file_name=file_name,
        ids=ids
    )

    return ids

In [ ]:
sqlite_insert_multiple_row_df(
    json_normalize_dict={
        "data": contents["messages"],
        "sep": "_"
    },
    mapping={
        "id": "message_id",
        "type": "message_type",
        "timestamp": "message_timestamp",
        "content": "message_content",
        "author_id": "author_id",
        "guild_id": "guild_id",
        "channel_id": "channel_id"
    },
    constant_cols={
        "guild_id": guild_id,
        "channel_id": channel_id
    },
    table_primary_key="message_id",
    table_name='discord_messages',
    conn=conn,
    file_name=json_file
)

## discord_authors

The `discord_authors` table has a unique shape.

The `id` column is what *must* be unique within the table, but we *care* about the uniqueness at the following combination:
- UNIQUE(author_id, author_name)

The *combination* of author_id and author_name is what *acutally* generates the uniqueness of the inserted element. This means we need a separate method for this kind of checking. The pseudo-code is as follows:
- author_id not present within the table? We want to insert all of these values
- author_id present within the table?
    - We must check to see if the *combination* of author_id and author_name is present
        - If no? Insert the data
        - If yes? Skip the insert, that data is present

In [ ]:
db = {
    "id": [1,2,3,4]
    ,"Unique_1":['A','B','C','C']
    ,"Unique_2":['Z','Y','X','W']
}

new_info = {
    "id": [5,6,7,8,9]
    ,"Unique_1":['D','A','A','B','A']
    ,"Unique_2":['V','U','I','C','Z']
}

db_df = pd.DataFrame(db)
new_df = pd.DataFrame(new_info)

print("New Unique_1's to Insert")
print(
    new_df[
        ~new_df['Unique_1'].isin(db['Unique_1'])
    ]
)

rest_of_df = new_df[
    new_df['Unique_1'].isin(db['Unique_1'])
]

unique_cols = ['Unique_1','Unique_2']
db_df = db_df[unique_cols]
rest_of_df = rest_of_df[unique_cols]

difference_df = rest_of_df[~rest_of_df.apply(tuple,1).isin(db_df.apply(tuple,1))]

print(difference_df.head())

In [24]:
import pandas as pd

db = {
    "id": [1,2,3,4]
    ,"Unique_1":['A','B','C','C']
    ,"Unique_2":['Z','Y','X','W']
}

db_df = pd.DataFrame(db)

uniques = [a+b for a in db_df["Unique_1"].tolist() for b in db_df["Unique_2"].tolist()]

print(uniques)

['AZ', 'AY', 'AX', 'AW', 'BZ', 'BY', 'BX', 'BW', 'CZ', 'CY', 'CX', 'CW', 'CZ', 'CY', 'CX', 'CW']


In [60]:
def character_separated_list(l: List[str], c=","):

    return f"{c}".join(l)

def all_from_table(table: str, conn: sqlite3.Connection) -> pd.DataFrame:

    select_sql = f"""
        SELECT 
            *
        FROM {table}
    """

    df = pd.read_sql_query(sql=select_sql,con=conn)

    return df

def cols_from_table(table: str, cols: List[str], conn: sqlite3.Connection) -> pd.DataFrame:

    select_sql = f"""
        SELECT 
            {character_separated_list(l=cols,c=",")}
        FROM {table}
    """

    df = pd.read_sql_query(sql=select_sql,con=conn)

    return df

# TODO: Remove overlap betwen this and single row and think about naming updates and readability for how many methods there are
def sqlite_insert_multiple_uniqueness_df(
        json_normalize_dict: dict
        , mapping: dict
        , table_name: str
        , first_level_unique: str
        , second_level_unique: str
        , conn: sqlite3.Connection
        , file_name: str
        , drop_col: str
        , constant_cols={}
        , drop_duplicates=False
    ) -> None:

    df = pd.json_normalize(**json_normalize_dict)

    if df.empty:
        print("No Records to Insert.")
        return -1

    df = assign_constant_column_values(df=df, constant_cols=constant_cols)

    # Filter DataFrame to only be mapped columns, if the col doesn't exist in the Database we are not interested.
    df = df[mapping.keys()]

    df.rename(columns=mapping, inplace=True)

    df = df[df[first_level_unique].notna()]
    df = df[df[second_level_unique].notna()]

    # Get DB data for any IDs that are NOT present
    db_data = all_from_table(
        table=table_name,
        conn=conn,
    )

    db_data.drop(drop_col, axis=1, inplace=True)

    if drop_duplicates:
        df = df.drop_duplicates(subset=[first_level_unique,second_level_unique])

    # Database table has NO data, fill it
    if db_data.empty:
        pass

    new_first_level_df = df[
        ~df[first_level_unique].isin(db_data[first_level_unique])
    ]

    rest_of_df = df[
        df[first_level_unique].isin(db_data[first_level_unique])
    ]

    difference_df = rest_of_df[~rest_of_df.apply(tuple,1).isin(db_data.apply(tuple,1))]

    insert_df = pd.concat([new_first_level_df,difference_df])

    test = insert_df[[first_level_unique, second_level_unique]].values.tolist()

    ids = [a[0]+a[1] for a in test]

    insert_df_to_db(
        df=insert_df,
        conn=conn,
        table_name=table_name,
        logging_table='discord_logging',
        file_name=file_name,
        ids=ids
    )

    return ids

In [ ]:
sqlite_insert_multiple_uniqueness_df(
    # Because of json_normalize, authors are at the messages level
    json_normalize_dict={
        "data": contents["messages"],
        "sep": "_"
    },
    mapping={
        "author_id": "author_id",
        "author_name": "author_name",
        "author_isBot": "author_is_bot",
        "author_avatarUrl": "author_avatar_URL",
    },
    first_level_unique="author_id",
    second_level_unique="author_name",
    table_name='discord_authors',
    conn=conn,
    file_name=json_file,
    drop_col="id",
    drop_duplicates=True
)

## discord_attachments

In [72]:
sqlite_insert_multiple_row_df(
    json_normalize_dict={
        "data": contents["messages"],
        "record_path": "attachments",
        "meta": ["id"],
        "record_prefix": "attachment_",
        "sep": "_"
    },
    mapping={
        "attachment_id": "attachment_id",
        "attachment_url": "attachment_URL",
        "attachment_fileName": "attachment_file_name",
        "attachment_fileSizeBytes": "attachment_file_size_bytes",
        "id": "message_id"
    },
    table_primary_key="attachment_id",
    table_name='discord_attachments',
    conn=conn,
    file_name=json_file
)


Nothing to insert.


['1264261784966135979',
 '1264261786111311964',
 '1264261787273003050',
 '1266877496171958417',
 '1275435357185245216',
 '1283831083594547220',
 '1283849978476892210',
 '1283871460875833344',
 '1285065443387899966',
 '1285092937255882833',
 '1285092937616855085',
 '1285092937922904166',
 '1286349438411083776',
 '1287417277142007849',
 '1287417277821751388',
 '1299910889750925415',
 '1303805405083996240']

## discord_stickers

In [73]:
sqlite_insert_multiple_row_df(
    json_normalize_dict={
        "data": contents["messages"],
        "record_path": "attachments",
        "meta": ["id"],
        "record_prefix": "sticker_",
        "sep": "_"
    },
    mapping={
        "sticker_id": "sticker_id",
        "sticker_fileName": "sticker_name",
        "sticker_url": "sticker_URL",
        "id": "message_id"
    },
    table_primary_key="sticker_id",
    table_name='discord_stickers',
    conn=conn,
    file_name=json_file
)


Nothing to insert.


['1264261784966135979',
 '1264261786111311964',
 '1264261787273003050',
 '1266877496171958417',
 '1275435357185245216',
 '1283831083594547220',
 '1283849978476892210',
 '1283871460875833344',
 '1285065443387899966',
 '1285092937255882833',
 '1285092937616855085',
 '1285092937922904166',
 '1286349438411083776',
 '1287417277142007849',
 '1287417277821751388',
 '1299910889750925415',
 '1303805405083996240']

## discord_mentions

In [ ]:
sqlite_insert_multiple_uniqueness_df(
    json_normalize_dict={
        "data": contents["messages"],
        "record_path": "mentions",
        "meta": ["id"],
        "record_prefix": "mention_",
        "sep": "_"
    },
    mapping={
        "mention_id": "author_id",
        "id": "message_id"
    },
    first_level_unique="author_id",
    second_level_unique="message_id",
    table_name='discord_mentions',
    conn=conn,
    file_name=json_file,
    drop_col="mention_id",
    drop_duplicates=True
)

## discord_embeds

In [ ]:
sqlite_insert_multiple_uniqueness_df(
    json_normalize_dict={
        "data": contents["messages"],
        "record_path": "embeds",
        "meta": ["id"],
        "sep": "_"
    },
    mapping={
        "title": "embed_title",
        "url": "embed_URL",
        "description": "embed_description",
        "author_name": "embed_author_name",
        "author_url": "embed_author_URL",
        "thumbnail_url": "embed_thumbnail_URL",
        "id": "message_id"
    },
    first_level_unique="message_id",
    second_level_unique="embed_URL",
    table_name='discord_embeds',
    conn=conn,
    file_name=json_file,
    drop_col="embed_id",
    drop_duplicates=True
)

['1250190703070416996https://www.youtube.com/watch?v=zjEeyxp7SKM',
 '1256979387346190469https://www.youtube.com/watch?v=kZW1BXCwODI',
 '1257681279672123453https://open.spotify.com/album/0bI3iaA6A68DTRcUoOyQx5?si=AmKqQaTQRPWppnzTxely7g',
 '1260979688038203407https://www.youtube.com/watch?v=uH7IyID5luc',
 '1261163244115071030https://www.youtube.com/watch?v=b7Bj1dBMYBE',
 '1261338624893521991https://open.spotify.com/album/3jBS0MSqzcUDOmaWfn4gZL?si=OLNw20BcQYOnP-SJsagYBQ',
 '1261340444709748787https://open.spotify.com/album/5DnmdbyeSEqiPGi0cbwqu8?si=m8KRLg9LQM6BmDM_Vok2EQ',
 '1266877495706255421https://www.youtube.com/watch?v=2TPfE73W5LE',
 '1266934112468598839https://www.youtube.com/watch?v=-W2bIDiB5jQ',
 '1270063727630094408https://open.spotify.com/playlist/6W5Pl7OSG1qcuv4iysdgMw?si=mVQQ7HwtScWZvTufoKGvdg&pi=zi-pmhYBRVu76',
 '1272989490230857914https://www.youtube.com/watch?v=gzMlcbKtgQM',
 '1273122907605831762https://strangerules.bandcamp.com/album/windowpane-acid',
 '127383569693894660

In [34]:
df = pd.json_normalize(
    data=contents["messages"],
    record_path="embeds",
    meta=["id"],
    sep="_"
)

df

,title,url,timestamp,description,color,images,fields,author_name,author_url,thumbnail_url,thumbnail_width,thumbnail_height,video_url,video_width,video_height,author_iconUrl,image_url,image_width,image_height,id
0,Midwife - Killdozer (Official Music Video),https://www.youtube.com/watch?v=zjEeyxp7SKM,None,Order No Depression in Heaven: https://flnsr.c...,#FF0000,[],[],The Flenser,https://www.youtube.com/channel/UCAyrTUtg9ZVTH...,https://images-ext-1.discordapp.net/external/I...,480.0,360.0,https://www.youtube.com/embed/zjEeyxp7SKM,981.0,720.0,NaN,NaN,NaN,NaN,1250190703070416996
1,One Last Thing (full album),https://www.youtube.com/watch?v=kZW1BXCwODI,None,Made by me\n\nReleased on: 2024-06-28\n\n1. Al...,#FF0000,[],[],Jules animations,https://www.youtube.com/channel/UCB9s5hlX7ysSt...,https://images-ext-1.discordapp.net/external/a...,1280.0,720.0,https://www.youtube.com/embed/kZW1BXCwODI,720.0,720.0,NaN,NaN,NaN,NaN,1256979387346190469
2,What Happened To The Heart?,https://open.spotify.com/album/0bI3iaA6A68DTRc...,None,AURORA · Album · 2024 · 16 songs.,NaN,[],[],NaN,NaN,https://images-ext-1.discordapp.net/external/c...,640.0,640.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1257681279672123453
3,Grouper - Disengaged + Heavy Water/Id Rather B...,https://www.youtube.com/watch?v=uH7IyID5luc,None,first and second songs of the album Dragging A...,#FF0000,[],[],tomeito,https://www.youtube.com/channel/UCaTSZdOvnIklt...,https://images-ext-1.discordapp.net/external/_...,480.0,360.0,https://www.youtube.com/embed/uH7IyID5luc,960.0,720.0,NaN,NaN,NaN,NaN,1260979688038203407
4,The Northern Boys - Nobody Likes Me,https://www.youtube.com/watch?v=b7Bj1dBMYBE,None,Listen to the song on Spotify: https://open.sp...,#FF0000,[],[],Sindhu World,https://www.youtube.com/channel/UCs4xFzPXJalS6...,https://images-ext-1.discordapp.net/external/S...,1280.0,720.0,https://www.youtube.com/embed/b7Bj1dBMYBE,1280.0,640.0,NaN,NaN,NaN,NaN,1261163244115071030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,Hum - Stars [HQ],https://www.youtube.com/watch?v=tIG2hRt-yec,None,,#FF0000,[],[],LISTENING PARTY,https://www.youtube.com/channel/UC3YlMEmaMJVYP...,https://images-ext-1.discordapp.net/external/U...,640.0,480.0,https://www.youtube.com/embed/tIG2hRt-yec,960.0,720.0,NaN,NaN,NaN,NaN,1304499302877364377
148,Early Snow (Acoustic),https://www.youtube.com/watch?v=zQCjESq5Vdk,None,Provided to YouTube by Universal Music Group\n...,#FF0000,[],[],The Sword - Topic,https://www.youtube.com/channel/UCTLDFfoaS5oG3...,https://images-ext-1.discordapp.net/external/I...,1280.0,720.0,https://www.youtube.com/embed/zQCjESq5Vdk,720.0,720.0,NaN,NaN,NaN,NaN,1304505615158214677
149,My Morning Jacket - Compound Fracture,https://www.youtube.com/watch?v=3LGDI_oz1Dg,None,Official video of My Morning Jacket’s “Compoun...,#FF0000,[],[],MyMorningJacketVEVO,https://www.youtube.com/channel/UCW_RJZNZMbZfO...,https://images-ext-1.discordapp.net/external/E...,1280.0,720.0,https://www.youtube.com/embed/3LGDI_oz1Dg,1280.0,720.0,NaN,NaN,NaN,NaN,1304507496479723520
150,My Morning Jacket - Like A River,https://www.youtube.com/watch?v=IZHNZaWIPis,None,"From My Morning Jacket's album ""The Waterfall""...",#FF0000,[],[],Callum Christie,https://www.youtube.com/channel/UClZlkfX0Kfie4...,https://images-ext-1.discordapp.net/external/p...,640.0,480.0,https://www.youtube.com/embed/IZHNZaWIPis,1280.0,719.0,NaN,NaN,NaN,NaN,1304508893484810240
